## 0. Описание лабораторной работы
**Задание:**

Набор представляет собой сезонные временные ряды с месячными данными по числу сотрудников в индустрии гостеприимства штата Калифорния. Данные отражают количество работников (в тысячах человек) по месяцам.

**Этапы выполнения:**

1. Предварительный анализ и визуализация данных
    * Загрузить данные и построить графики временных рядов занятости
    * Определить признаки сезонности, тренда и возможные аномалии
2. Проверка стационарности
    * Выполнить тесты для проверки стационарности ряда
    * При необходимости выполнить преобразования
3. Построение модели прогнозирования
    * Построить и обучить модель прогнозирования временных рядов
    * Сделать прогноз на несколько периодов
    * Оценить точность прогноза с помощью метрик MAE, RMSE
4. Анализ и интерпретация результатов
    * Визуализировать полученный прогноз вместе с исходным рядом
    * Интерпретировать выявленные сезонные и трендовые компоненты
    * Обсудить влияние сезонности на прогноз

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
import plotly.express as px
pio.templates.default = 'plotly_white'
import plotly.subplots as sp
import plotly.graph_objects as go

import pandas as pd
import numpy as np
import random
import sklearn
import os

seed = 42
random.seed(seed)
np.random.seed(seed)
sklearn.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

## 1. Предварительный анализ и визуализация данных


### 1.1. Загрузка данных

In [ ]:
data_url = 'https://github.com/Lopa10ko/itmo-ml-2025/raw/main/lab-5/california_hospitality.csv'

df = pd.read_csv(data_url, parse_dates=['Date'], index_col='Date')
df.index = pd.date_range(start='1990-01-01', periods=len(df), freq='MS')

print(f'Data time range: {df.index.min()} to {df.index.max()}')
print(f'Frequency: {df.index.freq}')
print(f'Shape: {df.shape}')

Data time range: 1990-01-01 00:00:00 to 2018-12-01 00:00:00
Frequency: <MonthBegin>
Shape: (348, 1)


Имеем дело с игрушечным датасетом.

Мощность ряда мала, но это ожидаемо, ведь имеем дело с месячной сезонностью.

In [ ]:
display(df.head())
print(f'\n\nОписательная статистика:')
display(df.describe())


,Employees
1990-01-01,1064.5
1990-02-01,1074.5
1990-03-01,1090.0
1990-04-01,1097.4
1990-05-01,1108.7




Описательная статистика:


,Employees
count,348.000000
mean,1452.506897
std,256.604914
min,1064.500000
25%,1238.050000
50%,1436.200000
75%,1586.300000
max,2022.100000


### 1.2. Построение графиков временных рядов занятости c компоненатами тренда и сезонности

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(df['Employees'], model='additive', period=12)

fig = sp.make_subplots(rows=3, cols=1)

fig.add_trace(go.Scatter(x=df.index, y=decomposition.observed, name='ts'), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.trend, name='trend',
                        line=dict(color='rgba(255, 0, 0, 0.7)')), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.seasonal, name='seasonal'), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.resid, name='residuals'), row=3, col=1)

fig.update_layout(height=1000)
fig.show()

Что хочется сразу отметить?

Наблюдаем ярко выраженный положительный тренд (почти линейный) и годичную сезонность.

Видимо, индустрия гостеприимства в Калифорнии, требует все больше и больше сотрудников с каждым годом.

Пики занятости приходятся на летние месяцы и периоды праздников, а спады наблюдаются в межсезонье. При этом амплитуда сезонных колебаний увеличивается по мере общего роста численности персонала, что характерно для растущих рынков гостеприимства.

Проверим на всякий случай, встречаются ли пропуски в значениях временного ряда:

In [ ]:
assert df['Employees'].isna().sum() == 0, 'Missing values detected'

## 2. Проверка стационарности


Для определения стационарности ряда проведем два стат.теста:
1. Расширенный тест Дики-Фуллера (Augmented Dickey-Fuller Test)
2. Тест Квятковского-Филлипса-Шмидта-Шина (Kwiatkowski-Phillips-Schmidt-Shin Test)

Намеренно выбрал удвоенный годичный период, чтобы немного уточнить тест и брать авторегрессию в окне из показаний последних двух периодов


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def test_stationarity(timeseries: pd.Series, alpha: float = 0.05, verbose: bool = False):
    adf_result = adfuller(timeseries, maxlag=12 * 2, regression='ct')
    kpss_result = kpss(timeseries, regression='ct')
    if verbose:
        print(f'ADF: {adf_result[0]:.4f}, p-value: {adf_result[1]:.4f}')
        print(f'KPSS: {kpss_result[0]:.4f}, p-value: {kpss_result[1]:.4f}')

    return kpss_result[1] >= alpha and adf_result[1] <= alpha, adf_result[1]

In [ ]:
is_stationary, _ = test_stationarity(df['Employees'], alpha=0.05, verbose=True)
if is_stationary:
    print('STATIONARY')
else:
    print('NON-STATIONARY')

ADF: -2.0948, p-value: 0.5490
KPSS: 0.3270, p-value: 0.0100
NON-STATIONARY


/tmp/ipython-input-1429272390.py:6: InterpolationWarning:

The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.




In [ ]:
from statsmodels.tsa.stattools import adfuller

def find_best_transformation(methods, seasonal_period=12, alpha=0.05):
    best_method = None
    best_p_value = 1.0
    best_series = None
    results = {}

    for name, series in methods.items():
        p_value = adfuller(series.dropna())[1]
        results[name] = {
            'series': series,
            'p_value': p_value,
            'stationary': 'stationary' if p_value <= alpha else 'non_stationary'
        }
        if p_value < best_p_value:
            best_p_value = p_value
            best_method = name
            best_series = series

    return results, best_method, best_series

def plot_transformations(transformations, original_df):
    sort_ts = sorted([(name, data) for name, data in transformations.items()],
                       key=lambda x: x[1]['p_value'])

    n_plots = len(sort_ts) + 1
    titles = ['Original'] + [f"{n} (p={d['p_value']:.4f}) - {d['stationary']}" for n, d in sort_ts]
    fig = sp.make_subplots(
        rows=n_plots, cols=1,
        subplot_titles=titles,
    )

    fig.add_trace(go.Scatter(x=original_df.index, y=original_df, name='Original'),
                 row=1, col=1)

    for i, (name, data) in enumerate(sort_ts, 2):
        series = data['series']
        fig.add_trace(go.Scatter(x=series.index, y=series, name=name), row=i, col=1)

    fig.update_layout(height=250 * n_plots, showlegend=False)
    fig.show()

Опишу методы преобразования временного ряда


1. Базовые преобразования
- **first_diff**: Первая разность $y'_t = y_t - y_{t-1}$
- **second_diff**: Вторая разность $y''_t = y_t - 2y_{t-1} + y_{t-2}$
- **seasonal_diff**: Сезонная разность $y'_t = y_t - y_{t-s}$, где $s$ - сезонный период

2. Логарифмические преобразования
- **log_first_diff**: Логарифм + первая разность $\Delta \ln(y_t) = \ln(y_t) - \ln(y_{t-1}) = \ln\left(\frac{y_t}{y_{t-1}}\right)$
- **log_seasonal_diff**: Логарифм + сезонная разность $\Delta_s \ln(y_t) = \ln(y_t) - \ln(y_{t-s}) = \ln\left(\frac{y_t}{y_{t-s}}\right)$

3. Степенные преобразования
- **sqrt_first_diff**: Квадратный корень + первая разность $\Delta \sqrt{y_t} = \sqrt{y_t} - \sqrt{y_{t-1}}$

4. Статистические преобразования
- **boxcox_diff**: Преобразование Бокса-Кокса + первая разность
  $$y_t^{(\lambda)} = \begin{cases}
  \frac{y_t^\lambda - 1}{\lambda} & \text{если } \lambda \neq 0 \\
  \ln(y_t) & \text{если } \lambda = 0
  \end{cases}$$
- **yeojohnson_diff**: Преобразование Йео-Джонсона + первая разность
  $$y_t^{(\lambda)} = \begin{cases}
  \frac{(y_t+1)^\lambda - 1}{\lambda} & \text{если } \lambda \neq 0, y_t \geq 0 \\
  \ln(y_t+1) & \text{если } \lambda = 0, y_t \geq 0 \\
  \frac{-(1-y_t)^{2-\lambda} - 1}{2-\lambda} & \text{если } \lambda \neq 2, y_t < 0 \\
  -\ln(1-y_t) & \text{если } \lambda = 2, y_t < 0
  \end{cases}$$

5. Корректировки тренда и сезонности
- **detrended**: Линейное устранение тренда $$y_t^{detrended} = y_t - (at + b)$$
- **residual**: Остаток от сезонной декомпозиции $$residual_t = y_t - trend_t - seasonal_t$$
- **rolling_adj**: Корректировка скользящим средним $$y_t^{adj} = y_t - \frac{1}{w}\sum_{i=t-w+1}^t y_i$$ где $w$ - размер окна

In [ ]:
from scipy import stats
from scipy import signal
from statsmodels.tsa.seasonal import seasonal_decompose

original_series = df['Employees']
seasonal_period = 12

boxcox_data, _ = stats.boxcox(original_series)
boxcox_series = pd.Series(boxcox_data, index=original_series.index)
boxcox_diff = boxcox_series.diff()

yj_data, _ = stats.yeojohnson(original_series)
yeojohnson_series = pd.Series(yj_data, index=original_series.index)
yeojohnson_diff = yeojohnson_series.diff()

decomposition = seasonal_decompose(original_series, period=min(seasonal_period, len(original_series)//2))
residual_series = decomposition.resid

rolling_mean = original_series.rolling(window=min(seasonal_period, len(original_series)), min_periods=1).mean()
rolling_mean_adj = (original_series - rolling_mean)

methods = {
    'first_diff': original_series.diff().dropna(),
    'second_diff': original_series.diff().diff().dropna(),
    'seasonal_diff': original_series.diff(seasonal_period).dropna(),
    'log_first_diff': np.log(original_series).diff().dropna(),
    'log_seasonal_diff': np.log(original_series).diff(seasonal_period).dropna(),
    'sqrt_first_diff': np.sqrt(original_series).diff().dropna(),
    'boxcox_diff': boxcox_diff.dropna(),
    'yeojohnson_diff': yeojohnson_diff.dropna(),
    'detrended': pd.Series(signal.detrend(original_series),
                           index=original_series.index).dropna(),
    'residual': residual_series.dropna(),
    'rolling_adj': rolling_mean_adj.dropna(),
}

results, best_method, best_series = find_best_transformation(methods)
plot_transformations(results, original_series)
print(f'Best transformation: {best_method}')

Best transformation: residual


Судя по сравнению методов трансформации временного ряда, можно определиться с методами.

Например, все предложенные методы хорошо себя показали в устранении тренда, однако для устранения и тренда и сезонности лучше применять:
- `log_seasonal_diff` - сохраняет относительные изменения, идеально для мультипликативной сезонности
- `residual` - полностью удаляет и тренд, и сезонность, оставляя только случайную компоненту

In [ ]:
stationary_series = np.log(df['Employees']).diff(12).dropna()

## 3. Построение модели прогнозирования

Построим форкастер и составим предсказания на 36 временных меток вперед (3 годичных периода)

In [ ]:
HORIZON = 36

metrics = {}
cv_results = {}
forecasts = {}
ts_train = stationary_series.iloc[:-HORIZON]
ts_test = stationary_series.iloc[-HORIZON:]
print(f"Training size: {len(ts_train)}, Test size: {len(ts_test)}")


Training size: 300, Test size: 36


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

def get_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    smape = 2.0 * np.mean(np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred))) * 100

    return {'rmse': rmse, 'mae': mae, 'smape': smape}

Я переборол желание запустить на данных Fedot.Industrial в autoML режиме.

Считаю это подвигом!

Решил ограничиться SARIMA с указанием годичной сезонности, наивным форкастером (повторением последней точки в исторических данных) и сезонным наивным форкастером (берутся значения последнего полного годичного цикла и повторяются в предсказании).

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def sarima_forecaster(train_series, horizon):
    model = SARIMAX(train_series, order=(1,0,0), seasonal_order=(1,0,0,12),
                          enforce_stationarity=False, enforce_invertibility=False)
    fitted_model = model.fit(disp=False, maxiter=100)
    forecast = fitted_model.forecast(steps=horizon)
    return forecast.values

def naive_forecaster(train_series, horizon):
    last_value = train_series.iloc[-1]
    return np.full(horizon, last_value)

def seasonal_naive_forecaster(train_series, horizon, seasonality=12):
    if len(train_series) >= seasonality:
        seasonal_values = train_series.values[-seasonality:]
        full_cycles = horizon // seasonality
        remainder = horizon % seasonality
        forecast = np.tile(seasonal_values, full_cycles)
        forecast = np.concatenate([forecast, seasonal_values[:remainder]])
    else:
        forecast = naive_forecaster(train_series, horizon)
    return forecast

In [ ]:
forecasters = {
    'sarima': sarima_forecaster,
    'naive': naive_forecaster,
    'seasonal_naive': lambda train, hor: seasonal_naive_forecaster(train, hor, 12)
}

for name, forecaster_func in forecasters.items():
    print(f'Training {name}...')
    final_forecast = forecaster_func(ts_train, HORIZON)

    forecasts[name] = final_forecast
    metrics[name] = get_metrics(ts_test.values, final_forecast)

Training sarima...
Training naive...
Training seasonal_naive...


В погоне за хайпом запустим zero-shot форкастер на трансформере

In [ ]:
!pip install git+https://github.com/amazon-science/chronos-forecasting.git --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 3.3 MB/s eta 0:00:00


In [ ]:
from chronos import BaseChronosPipeline
import torch

HF_MODEL = 'amazon/chronos-t5-tiny'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

chronos_forecaster = BaseChronosPipeline.from_pretrained(HF_MODEL,
                                                         dtype=DTYPE,
                                                         device_map=DEVICE)
chronos_forecast = chronos_forecaster.predict(torch.tensor(ts_train.values),
                                              prediction_length=HORIZON)
chronos_forecast = np.median(chronos_forecast[0].numpy(), axis=0)

metrics['chronos'] = get_metrics(ts_test, chronos_forecast)
forecasts['chronos'] = chronos_forecast

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

## 4. Анализ и интерпретация результатов

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import prettytable as pt
import numpy as np

def plot_forecasts_comparison(ts, forecasts, metrics, horizon):
    fig = make_subplots(rows=2, cols=1, row_heights=[0.7, 0.3])

    n = len(ts)
    train_end = n - horizon
    test_indices = list(range(train_end, n))

    fig.add_trace(go.Scatter(x=list(range(n)), y=ts, mode='lines', name='Actual',
                            line=dict(color='black', width=2), opacity=0.7), row=1, col=1)

    for i, (name, forecast) in enumerate(forecasts.items()):
        fig.add_trace(go.Scatter(x=test_indices, y=forecast, mode='lines+markers',
                                name=f'{name} Forecast', marker=dict(size=4)), row=1, col=1)

        actual_test = ts[train_end:]
        error = forecast - actual_test
        fig.add_trace(go.Scatter(x=test_indices, y=error,
                                mode='lines', name=f'{name} Error',
                                showlegend=False), row=2, col=1)

    fig.update_layout(height=800, showlegend=True, xaxis_title='Time Index',
                     yaxis_title='Value', xaxis2_title='Time Index',
                     yaxis2_title='Forecast Error')
    fig.add_vline(x=train_end, line_dash="dash", line_color="gray",
                  annotation_text="Train/Test Split", row=1, col=1)
    fig.show()

plot_forecasts_comparison(stationary_series, forecasts, metrics, HORIZON)

In [ ]:
from prettytable import PrettyTable

def create_metrics_table(metrics):
    first_forecaster = list(metrics.keys())[0]
    metric_names = list(metrics[first_forecaster].keys())

    table = PrettyTable()
    table.field_names = ['Model'] + metric_names

    for name, metric_dict in metrics.items():
        row = [name]
        for metric_name in metric_names:
            value = metric_dict.get(metric_name, np.nan)
            row.append(f"{value:.4f}")
        table.add_row(row)

    if metric_names:
        table.sortby = metric_names[0]

    return table

print(create_metrics_table(metrics))

+----------------+--------+--------+---------+
|     Model      |  rmse  |  mae   |  smape  |
+----------------+--------+--------+---------+
|    chronos     | 0.0070 | 0.0059 | 23.7150 |
|     sarima     | 0.0157 | 0.0133 | 43.7735 |
| seasonal_naive | 0.0162 | 0.0134 | 43.9890 |
|     naive      | 0.0194 | 0.0167 | 51.0968 |
+----------------+--------+--------+---------+


Не стал тюнить модели, но в целом LLM-like форкастер показал лучший результат, что ожидаемо для маломощных временных рядов и относительно длинного горизонта предсказаний.


Для улучшения прогноза ARIMA-модели можно использовать рекуррентный прогноз (recursive forecasting), при котором на каждом шаге предсказанное значение добавляется в историю, и модель пересчитывается для получения следующего прогноза. Однако этот подход вычислительно затратен и может приводить к накоплению ошибки.

Построим также график для нетрансформированных данных и предсказаний моделей на тестовом поднаборе

In [ ]:
def reverse_log_diff_transformations(stationary_forecasts, original_series, horizon):
    original_train = original_series.iloc[:-horizon]
    log_series = np.log(original_series)
    diff_series = log_series.diff(12)

    last_log_values = log_series.iloc[-horizon-12:-horizon].values
    log_forecasts_reconstructed = np.zeros(horizon)

    for i in range(horizon):
        if i < 12:
            log_forecasts_reconstructed[i] = last_log_values[i] + stationary_forecasts[i]
        else:
            log_forecasts_reconstructed[i] = log_forecasts_reconstructed[i-12] + stationary_forecasts[i]
    original_forecasts = np.exp(log_forecasts_reconstructed)

    return original_forecasts

original_forecasts = {}
for method, forecast in forecasts.items():
    original_forecasts[method] = reverse_log_diff_transformations(forecast, df['Employees'], HORIZON)

In [ ]:
plot_forecasts_comparison(df['Employees'], original_forecasts, metrics, HORIZON)

Сезонность в данных оказывает фундаментальное влияние на качество прогноза, так как технически ее наличие не позволяет предполагать стационарность процесса.

В наших данных конкретно был выявлен явный возрастающий тренд и сезонность каждые 12 значений (связана с тем, что паттерны процесса прослеживаются годично).

Если не трансформировать ряд и оставлять компоненты тренда и сезонности в тренировочных исторических данных, модель неверно будет интерпретировать сезонные пики и спады в качестве тенденций или шума.

Подобное поведение также можно заметить при неверно настроенных гиперпараметрах SARIMA, а прогноз в таком случае будет представлять из себя ярко выраженную возрастающую последовательность данных, вторя тренду.

**Лабораторная работа выполнена в рамках курса "Машинное обучение" ИТМО, 2025**